# CoTracker: full-video inference with lower GPU memory

Restart the kernel before running this notebook to release tensors from the offline demo. Select `dvss-pytorch`. This uses the online checkpoint and overlapping windows with persistent tracking state; predictions can differ from offline inference. Video stays on CPU as uint8.


In [6]:
from pathlib import Path
import sys

# Jupyter may start in either the repository root or notebooks/.
root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "cotracker" / "predictor.py").is_file()),
    None,
)
if root is None:
    raise RuntimeError(
        "Cannot locate the co-tracker repository. Open this notebook from "
        "the co-tracker project folder and restart the kernel."
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print(f"Python: {sys.executable}")
print(f"Repository: {root}")

import torch
from cotracker.predictor import CoTrackerOnlinePredictor
from cotracker.utils.visualizer import Visualizer, read_video_from_path

assert torch.cuda.is_available(), "Select the CUDA-enabled dvss-pytorch kernel."
device = torch.device("cuda:0")
checkpoint = root / "checkpoints" / "scaled_online.pth"
assert checkpoint.is_file(), f"Missing online checkpoint: {checkpoint}"
model = CoTrackerOnlinePredictor(checkpoint=checkpoint.as_posix()).to(device).eval()
print(torch.cuda.get_device_name(device))


Python: c:\Users\yxl3527\.conda\envs\dvss-pytorch\python.exe
Repository: c:\Yizhou Li Folder\github_project\co-tracker
NVIDIA GeForce RTX 4090 Laptop GPU


In [8]:
video_path = r"C:/ros2/ros_ws/src/varjo_ros2_example/RealData/User99/Video/test2.mp4"
# Keep all frames in CPU memory; convert only the current window to float32 on GPU.
video = torch.from_numpy(read_video_from_path(video_path)).permute(0, 3, 1, 2)[None]
assert video.device.type == "cpu" and video.dtype == torch.uint8
print(f"Video: {tuple(video.shape)}, {video.device}, {video.dtype}")


Video: (1, 900, 3, 480, 640), cpu, torch.uint8


## Select tracking points with the mouse
Run the next cell to open the first frame in a separate window (it may appear behind VS Code). Left-click to add points, right-click or Backspace to undo, Enter to accept, Esc to cancel. Pixel coordinates refer to the original frame, with (0, 0) at the top-left. Rerun this cell to choose new points, then rerun inference and visualization.


In [9]:
import tkinter as tk
from PIL import Image, ImageTk


def select_tracking_points(frame):
    """Return original-resolution (x, y) pixels selected in an RGB frame."""
    height, width = frame.shape[:2]
    window = tk.Tk()
    window.title("Select points: left-click add | right-click undo | Enter accept | Esc cancel")
    scale = min(1.0, (window.winfo_screenwidth() - 100) / width,
                (window.winfo_screenheight() - 180) / height)
    display_width = max(1, int(width * scale))
    display_height = max(1, int(height * scale))
    picture = ImageTk.PhotoImage(Image.fromarray(frame).resize((display_width, display_height)))
    canvas = tk.Canvas(window, width=display_width, height=display_height,
                       highlightthickness=0)
    canvas.pack()
    status = tk.Label(window, text="Click points, then press Enter (at least one point).")
    status.pack()
    points = []
    accepted = False

    def redraw():
        canvas.delete("all")
        canvas.create_image(0, 0, image=picture, anchor="nw")
        for index, (x, y) in enumerate(points):
            cx, cy = x * display_width / width, y * display_height / height
            canvas.create_oval(cx - 4, cy - 4, cx + 4, cy + 4,
                               fill="red", outline="white")
            canvas.create_text(cx + 8, cy - 8, text=str(index + 1),
                               fill="yellow", anchor="sw")
        status.config(text=f"{len(points)} points selected. Enter: accept | Backspace: undo | Esc: cancel")

    def add(event):
        x = min(width - 1, max(0, round(event.x * width / display_width)))
        y = min(height - 1, max(0, round(event.y * height / display_height)))
        points.append((x, y))
        redraw()

    def undo(event=None):
        if points:
            points.pop()
            redraw()

    def accept(event=None):
        nonlocal accepted
        if points:
            accepted = True
            window.destroy()

    canvas.bind("<Button-1>", add)
    canvas.bind("<Button-3>", undo)
    window.bind("<BackSpace>", undo)
    window.bind("<Return>", accept)
    window.bind("<Escape>", lambda event: window.destroy())
    window.protocol("WM_DELETE_WINDOW", window.destroy)
    redraw()
    window.lift()
    canvas.focus_set()
    try:
        window.mainloop()
    finally:
        if tk._default_root is window:
            window.destroy()
    if not accepted:
        raise RuntimeError("Point selection cancelled. Rerun this cell to select points.")
    return points


# Clear previous queries so cancelling cannot reuse an old selection.
queries = None
selected_points = select_tracking_points(video[0, 0].permute(1, 2, 0).numpy())
# CoTracker expects (batch, points, 3), where each point is (frame, x, y).
queries = torch.tensor([[[0, x, y] for x, y in selected_points]],
                       dtype=torch.float32, device=device)
print("Selected (x, y) pixels:", selected_points)


Selected (x, y) pixels: [(193, 302), (237, 301), (211, 324), (223, 282), (197, 128), (201, 148)]


In [10]:
assert queries is not None and queries.shape[1] > 0, "Select points first."
num_frames = video.shape[1]
assert num_frames >= 2, "At least two video frames are required."
torch.cuda.reset_peak_memory_stats(device)

with torch.inference_mode():
    # The first call initializes queries and tracking state without running inference.
    model(video[:, :1].to(device=device, dtype=torch.float32),
          is_first_step=True, queries=queries, add_support_grid=True)
    start = 0
    while True:
        end = min(start + 2 * model.step, num_frames)
        chunk = video[:, start:end].to(device=device, dtype=torch.float32)
        pred_tracks, pred_visibility = model(chunk, add_support_grid=True)
        del chunk
        if end == num_frames:
            break
        start += model.step
    # Each online result contains the accumulated tracks; use the final result.
    pred_tracks = pred_tracks[:, :num_frames].cpu()
    pred_visibility = pred_visibility[:, :num_frames].cpu()

torch.cuda.synchronize(device)
print(f"Tracks: {tuple(pred_tracks.shape)}, {pred_tracks.device}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated(device) / 1024**3:.2f} GiB")


Tracks: (1, 900, 6, 2), cpu
Peak GPU memory: 1.53 GiB


In [11]:
# Match the source frame rate and avoid adding a first-frame pause.
import cv2
import math

capture = cv2.VideoCapture(str(video_path))
try:
    output_fps = capture.get(cv2.CAP_PROP_FPS)
finally:
    capture.release()
if not math.isfinite(output_fps) or output_fps <= 0:
    raise RuntimeError("Cannot read source FPS. Set output_fps manually (e.g. 60).")
print(f"Output: {output_fps:g} fps, {video.shape[1] / output_fps:.2f} seconds")

# Rendering uses CPU memory and may take time for a long video.
vis = Visualizer(save_dir=str(root / "videos"), pad_value=100,
                 fps=output_fps, show_first_frame=0)
vis.visualize(video=video, tracks=pred_tracks, visibility=pred_visibility,
              filename="teaser_online")


Output: 60 fps, 15.00 seconds


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (840, 680) to (848, 688) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Video saved to c:\Yizhou Li Folder\github_project\co-tracker\videos\teaser_online.mp4


tensor([[[[[255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           ...,
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255]],

          [[255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           ...,
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255]],

          [[255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           ...,
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255],
           [255, 255, 255,  ..., 255, 255, 255]]],


         [[[255, 255, 255,  ..., 255, 255, 255],
           [2